In [7]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import TimeDistributed, Dense, LSTM, Dropout, Conv2D, MaxPooling2D, Flatten, Input
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils import shuffle
from tensorflow.keras.utils import to_categorical

# Set constants
SEQ_LENGTH = 16  # Number of frames per video to use
IMG_SIZE = (90, 75)  # Height, Width
BATCH_SIZE = 16
EPOCHS = 20

In [8]:
def load_dataset(csv_file, frames_dir, max_samples=None):
    """
    Load data from CSV and associate with video frames
    
    Args:
        csv_file: Path to CSV file with metadata
        frames_dir: Directory containing frame folders
        max_samples: Maximum number of samples to load (for testing/development)
    
    Returns:
        DataFrame with video metadata
    """
    # Load CSV
    df = pd.read_csv(csv_file)
    
    # Limit samples if specified
    if max_samples:
        df = df.sample(min(max_samples, len(df)))
    
    print(f"Loaded {len(df)} samples from {csv_file}")
    return df

def preprocess_frame(frame, target_size=IMG_SIZE):
    """Preprocess a single frame"""
    # Resize frame
    frame = cv2.resize(frame, (target_size[1], target_size[0]))
    
    # Convert to RGB if it's in BGR
    if len(frame.shape) == 3 and frame.shape[2] == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Normalize pixel values
    frame = frame / 255.0
    
    return frame

def extract_hand_roi(frame, mp_hands, hands, mp_draw):
    """Extract hand region of interest using MediaPipe"""
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    
    if not result.multi_hand_landmarks:
        return preprocess_frame(frame)  # Return preprocessed full frame if no hands
    
    # Track coordinates for all detected hands
    all_x_min, all_y_min = float('inf'), float('inf')
    all_x_max, all_y_max = 0, 0
    
    custom_landmark_drawing_spec = mp_draw.DrawingSpec(
        color=(0, 255, 0),
        thickness=1,
        circle_radius=1
    )
    
    custom_connection_drawing_spec = mp_draw.DrawingSpec(
        color=(0, 0, 255),
        thickness=1,
        circle_radius=1
    )
    
    # Process all detected hands
    for hand_landmarks in result.multi_hand_landmarks:
        # Find boundaries for this hand
        x_min = min([lm.x for lm in hand_landmarks.landmark])
        y_min = min([lm.y for lm in hand_landmarks.landmark])
        x_max = max([lm.x for lm in hand_landmarks.landmark])
        y_max = max([lm.y for lm in hand_landmarks.landmark])
        
        # Update the overall boundaries
        all_x_min = min(all_x_min, x_min)
        all_y_min = min(all_y_min, y_min)
        all_x_max = max(all_x_max, x_max)
        all_y_max = max(all_y_max, y_max)
    
    # Convert normalized coordinates to pixel coordinates
    h, w = frame.shape[:2]
    all_x_min, all_x_max = max(0, int(all_x_min * w)), min(w, int(all_x_max * w))
    all_y_min, all_y_max = max(0, int(all_y_min * h)), min(h, int(all_y_max * h))
    
    # Add padding
    pad_x = int((all_x_max - all_x_min) * 0.2)
    pad_y = int((all_y_max - all_y_min) * 0.2)
    
    all_x_min = max(0, all_x_min - pad_x)
    all_y_min = max(0, all_y_min - pad_y)
    all_x_max = min(w, all_x_max + pad_x)
    all_y_max = min(h, all_y_max + pad_y)
    
    # Extract hand ROI
    roi = frame[all_y_min:all_y_max, all_x_min:all_x_max]
    
    # If ROI extraction failed, return the full frame
    if roi.size == 0:
        return preprocess_frame(frame)
    
    return preprocess_frame(roi)

def load_video_frames(video_id, frames_dir, num_frames=SEQ_LENGTH, use_mediapipe=False):
    """
    Load frames for a specific video and preprocess them
    
    Args:
        video_id: ID of the video
        frames_dir: Directory containing frame folders
        num_frames: Number of frames to sample from the video
        use_mediapipe: Whether to use MediaPipe for hand detection
    
    Returns:
        Numpy array of preprocessed frames
    """
    # Path to video frames folder
    video_dir = os.path.join(frames_dir, video_id)
    
    if not os.path.exists(video_dir):
        print(f"Warning: Video directory not found: {video_dir}")
        return None
    
    # List all frames in the video directory
    frame_files = sorted([f for f in os.listdir(video_dir) if f.endswith('.jpg') or f.endswith('.png')])
    
    if len(frame_files) == 0:
        print(f"Warning: No frames found in {video_dir}")
        return None
    
    # If we need MediaPipe for hand detection
    if use_mediapipe:
        import mediapipe as mp
        mp_hands = mp.solutions.hands
        hands = mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.5)
        mp_draw = mp.solutions.drawing_utils
    else:
        mp_hands = None
        hands = None
        mp_draw = None
    
    # Sample frames uniformly if we have more frames than needed
    if len(frame_files) > num_frames:
        # Get indices of frames to sample
        indices = np.linspace(0, len(frame_files) - 1, num_frames, dtype=int)
        sampled_frames = [frame_files[i] for i in indices]
    else:
        # If we don't have enough frames, repeat the last frame
        sampled_frames = frame_files + [frame_files[-1]] * (num_frames - len(frame_files))
        sampled_frames = sampled_frames[:num_frames]
    
    # Load and preprocess frames
    frames = []
    for frame_file in sampled_frames:
        frame_path = os.path.join(video_dir, frame_file)
        frame = cv2.imread(frame_path)
        
        if frame is None:
            print(f"Warning: Could not read frame: {frame_path}")
            # Create a blank frame if image can't be read
            frame = np.zeros((IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)
        
        # Process the frame (extract hand ROI if using MediaPipe)
        if use_mediapipe:
            processed_frame = extract_hand_roi(frame, mp_hands, hands, mp_draw)
        else:
            processed_frame = preprocess_frame(frame)
        
        frames.append(processed_frame)
    
    # Clean up MediaPipe resources
    if use_mediapipe and hands is not None:
        hands.close()
    
    return np.array(frames)

# def data_generator(df, frames_dir, batch_size=BATCH_SIZE, num_classes=27, use_mediapipe=False):
#     """
#     Generator that yields batches of video frames and labels
    
#     Args:
#         df: DataFrame with video metadata
#         frames_dir: Directory containing frame folders
#         batch_size: Batch size
#         num_classes: Number of classes for one-hot encoding
#         use_mediapipe: Whether to use MediaPipe for hand detection
    
#     Yields:
#         Tuple of (batch_x, batch_y) with videos and one-hot encoded labels
#     """
#     print(f"Starting data generator with {len(df)} samples")

#     while True:
#         # Shuffle the dataframe at the beginning of each epoch
#         df = shuffle(df)
        
#         # Process data in batches
#         for start_idx in range(0, len(df), batch_size):
#             batch_df = df.iloc[start_idx:start_idx + batch_size]
            
#             batch_x = []
#             batch_y = []
            
#             for _, row in batch_df.iterrows():
#                 video_id = str[row['video_id']]
#                 label_id = row['label_id']
                
#                 # Load video frames
#                 frames = load_video_frames(video_id, frames_dir, use_mediapipe=use_mediapipe)
                
#                 if frames is not None:
#                     batch_x.append(frames)
#                     batch_y.append(label_id)
            
#             if len(batch_x) > 0:
#                 # Convert to numpy arrays
#                 batch_x = np.array(batch_x)
#                 batch_y = to_categorical(np.array(batch_y), num_classes=num_classes)
                
#                 yield batch_x, batch_y

In [9]:
def data_generator(df, frames_dir, batch_size=BATCH_SIZE, num_classes=27, use_mediapipe=False):
    """
    Generator that yields batches of video frames and labels with debug printing
    """
    print(f"Starting data generator with {len(df)} samples")
    
    while True:
        # Shuffle the dataframe at the beginning of each epoch
        df = shuffle(df)
        
        # Process data in batches
        for start_idx in range(0, len(df), batch_size):
            end_idx = min(start_idx + batch_size, len(df))
            batch_df = df.iloc[start_idx:end_idx]
            
            print("----------------------------------------------------------------------\n")
            print(f"Processing batch from index {start_idx} to {end_idx-1} (size: {len(batch_df)})")
            
            batch_x = []
            batch_y = []
            
            for i, (_, row) in enumerate(batch_df.iterrows()):
                # Convert video_id to string to ensure it's compatible with os.path.join
                video_id = str(row['video_id'])
                
                try:
                    label_id = int(row['label_id'])
                except (ValueError, KeyError) as e:
                    print(f"Error with label_id for video {video_id}: {e}")
                    print(f"Row data: {row}")
                    continue
                
                print(f"  Loading video {i+1}/{len(batch_df)}: {video_id} (label: {label_id})")
                
                # Path to video frames folder
                video_dir = os.path.join(frames_dir, video_id)
                
                if not os.path.exists(video_dir):
                    print(f"  ❌ Video directory not found: {video_dir}")
                    continue
                
                # List all frames in the video directory
                try:
                    frame_files = sorted([f for f in os.listdir(video_dir) if f.endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
                    print(f"  Found {len(frame_files)} frame files in {video_dir}")
                    
                    if len(frame_files) == 0:
                        print(f"  ❌ No valid frame files found in {video_dir}")
                        continue
                    
                    # Sample frames uniformly if we have more frames than needed
                    if len(frame_files) > SEQ_LENGTH:
                        indices = np.linspace(0, len(frame_files) - 1, SEQ_LENGTH, dtype=int)
                        sampled_frames = [frame_files[i] for i in indices]
                        print(f"  ℹ️ Sampling {SEQ_LENGTH} frames from {len(frame_files)} available")
                    else:
                        # If we don't have enough frames, repeat the last frame
                        sampled_frames = frame_files + [frame_files[-1]] * (SEQ_LENGTH - len(frame_files))
                        sampled_frames = sampled_frames[:SEQ_LENGTH]
                        print(f"  ⚠️ Padding to {SEQ_LENGTH} frames (only {len(frame_files)} available)")
                    
                    # Load and preprocess frames
                    frames = []
                    for j, frame_file in enumerate(sampled_frames):
                        frame_path = os.path.join(video_dir, frame_file)
                        frame = cv2.imread(frame_path)
                        
                        if frame is None:
                            print(f"  ❌ Could not read frame: {frame_path}")
                            # Create a blank frame if image can't be read
                            frame = np.zeros((IMG_SIZE[0], IMG_SIZE[1], 3), dtype=np.uint8)
                        
                        # Preprocess the frame
                        processed_frame = preprocess_frame(frame)
                        frames.append(processed_frame)
                    
                    frames = np.array(frames)
                    print(f"  ✅ Successfully loaded {len(frames)} frames with shape {frames.shape}")
                    
                    batch_x.append(frames)
                    batch_y.append(label_id)
                    
                except Exception as e:
                    print(f"  ❌ Error processing video {video_id}: {e}")
                    import traceback
                    traceback.print_exc()
                    continue
            
            if len(batch_x) > 0:
                print(f"✅ Batch complete with {len(batch_x)} valid samples")
                # Convert to numpy arrays
                batch_x = np.array(batch_x)
                batch_y = to_categorical(np.array(batch_y), num_classes=num_classes)
                
                print(f"✅ Final batch shapes: X={batch_x.shape}, Y={batch_y.shape}")
                print("\n----------------------------------------------------------------------")
                
                yield batch_x, batch_y
            else:
                print(f"❌ No valid samples in batch - cannot yield empty batch")
                print("\n----------------------------------------------------------------------")
                
                # Create dummy batch as a last resort to avoid generator emptiness
                dummy_x = np.zeros((1, SEQ_LENGTH, IMG_SIZE[0], IMG_SIZE[1], 3))
                dummy_y = to_categorical([0], num_classes=num_classes)
                print(f"⚠️ Yielding dummy batch as fallback")
                yield dummy_x, dummy_y

In [10]:
def build_enhanced_inception_lstm_model(num_classes, seq_length=SEQ_LENGTH, img_size=IMG_SIZE):
    """
    Build an enhanced model with CNN and MaxPool layers before InceptionV3,
    followed by LSTM for sequence processing
    
    Args:
        num_classes: Number of output classes
        seq_length: Number of frames per video
        img_size: Size of each frame (height, width)
    
    Returns:
        Compiled model
    """
    from tensorflow.keras.layers import UpSampling2D, GlobalAveragePooling2D

    min_size = 75

    # Define input shape for sequence of frames
    sequence_input = Input(shape=(seq_length, img_size[0], img_size[1], 3))
    
    # First, process each frame with custom CNN layers
    # Create a preprocessing model that will be applied to each frame
    frame_input = Input(shape=(img_size[0], img_size[1], 3))
    
    if img_size[0] < min_size or img_size[1] < min_size:
        # Calculate upsampling factor to reach minimum size
        upsample_factor_h = max(1, min_size // img_size[0])
        upsample_factor_w = max(1, min_size // img_size[1])
        upsample_factor = max(upsample_factor_h, upsample_factor_w)
        
        # Apply upsampling if needed
        x = UpSampling2D(size=(upsample_factor, upsample_factor))(frame_input)
        
        # Add some CNN layers to improve the upsampled image quality
        x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
        x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    else:
        x = frame_input
    
    # Load InceptionV3 with weights
    base_model = InceptionV3(
        include_top=False, 
        weights='imagenet', 
        input_shape=(None, None, 3),
        pooling='avg'
    )
    base_model.trainable = False
    
    # Connect InceptionV3 to our preprocessing layers
    inception_features = base_model(x)
    
    # Create the frame processor model
    frame_processor = Model(inputs=frame_input, outputs=inception_features)
    
    # Apply the frame processor to each frame in the sequence
    encoded_frames = TimeDistributed(frame_processor)(sequence_input)
    
    # Add LSTM layers for sequence processing
    lstm_out = LSTM(512, return_sequences=True, dropout=0.2,activation="tanh")(encoded_frames)
    lstm_out = LSTM(256, return_sequences=False, dropout=0.2,activation="tanh")(lstm_out)
    
    # Add dense layers for classification
    x = Dense(256, activation='relu')(lstm_out)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    # Create and compile the complete model
    model = Model(inputs=sequence_input, outputs=outputs)
    
    # Use a lower learning rate for fine-tuning
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

In [11]:
# Set paths to dataset
base_dir = "/mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2"  # Change to your dataset path
train_dir = os.path.join(base_dir, "Train")
test_dir = os.path.join(base_dir, "Test")
val_dir = os.path.join(base_dir, "Validation")
train_csv = os.path.join(base_dir, "Train.csv")
val_csv = os.path.join(base_dir, "Validation.csv")
test_csv = os.path.join(base_dir, "Test.csv")

# Load datasets
train_df = load_dataset(train_csv, train_dir)
val_df = load_dataset(val_csv, val_dir)
test_df = load_dataset(test_csv, test_dir)

# Determine number of classes
num_classes = len(train_df['label_id'].unique())
print(f"Number of classes: {num_classes}")

# Create data generators
train_gen = data_generator(train_df, train_dir, batch_size=BATCH_SIZE, num_classes=num_classes)
val_gen = data_generator(val_df, val_dir, batch_size=BATCH_SIZE, num_classes=num_classes)

Loaded 50420 samples from /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Train.csv
Loaded 7047 samples from /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Validation.csv
Loaded 6981 samples from /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Test.csv
Number of classes: 27


In [ ]:
# Build model
model = build_enhanced_inception_lstm_model(num_classes)
    # Alternatively: model = build_inception_lstm_model(num_classes)
    
model.summary()
    
    # Define callbacks
checkpoint_cb = ModelCheckpoint(
        'gesture_model_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
    
early_stopping_cb = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Train model
steps_per_epoch = len(train_df) // BATCH_SIZE
validation_steps = len(val_df) // BATCH_SIZE

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

# Save final model
model.save('gesture_model_final.h5')

# Optionally evaluate on test set
test_gen = data_generator(test_df, test_dir, batch_size=BATCH_SIZE, num_classes=num_classes)
test_steps = len(test_df) // BATCH_SIZE

test_results = model.evaluate(test_gen, steps=test_steps)
print(f"Test loss: {test_results[0]}, Test accuracy: {test_results[1]}")

# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 16, 90, 75, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 2048)       │    21,802,784 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 16, 512)        │     5,244,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 256)            │       787,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 27)             │         6,939 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,907,899 (106.46 MB)

 Trainable params: 6,105,115 (23.29 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

Starting data generator with 50420 samples
----------------------------------------------------------------------

Processing batch from index 0 to 15 (size: 16)
  Loading video 1/16: 37241 (label: 4)
  Found 37 frame files in /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Train/37241
  ℹ️ Sampling 16 frames from 37 available
  ✅ Successfully loaded 16 frames with shape (16, 90, 75, 3)
  Loading video 2/16: 3852 (label: 12)
  Found 37 frame files in /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Train/3852
  ℹ️ Sampling 16 frames from 37 available
  ✅ Successfully loaded 16 frames with shape (16, 90, 75, 3)
  Loading video 3/16: 56463 (label: 15)
  Found 37 frame files in /mnt/MainDrive/Codes/Deep Learning/Gesture_control/3D_CNN+Lstm/archive_2/Train/56463
  ℹ️ Sampling 16 frames from 37 available
  ✅ Successfully loaded 16 frames with shape (16, 90, 75, 3)
  Loading video 4/16: 139575 (label: 0)
  Found 37 frame files in /mnt/MainDriv